# 06 - Safe Data Augmentation for ASL Videos

## Objective

Increase dataset diversity through augmentation while **preserving sign meaning**.

## Why ASL Augmentation is Different

In sign language, **geometry is grammar**:
- Hand orientation defines the word
- Direction of movement has meaning
- Temporal order cannot be reversed

### Dangerous Augmentations (AVOID)

| Augmentation | Why Dangerous |
|--------------|---------------|
| Horizontal Flip | Changes dominant hand, may change meaning |
| Strong Rotation (>15°) | Hand orientation IS the sign |
| Time Reversal | Destroys temporal meaning |

### Safe Augmentations (USE)

| Augmentation | Effect | Why Safe |
|--------------|--------|----------|
| Random Crop/Zoom | Spatial invariance | Hand position in frame doesn't define sign |
| Color Jitter | Lighting invariance | Skin tone/room lighting irrelevant |
| Frame Dropping | Temporal robustness | Real videos have variable FPS |
| Small Rotation (±5°) | Camera tilt | Slight camera angle doesn't change meaning |
| Gaussian Noise | Sensor noise | Quality variations are normal |

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 0. Environment Setup

In [ ]:
import os
import json
import shutil
from pathlib import Path
from typing import List, Tuple, Optional, Callable
from datetime import datetime
import random

import numpy as np
import torch
from PIL import Image, ImageEnhance, ImageFilter
import torchvision.transforms as T
import torchvision.transforms.functional as TF
from tqdm.auto import tqdm
import matplotlib.pyplot as plt

# Set seeds for reproducibility
random.seed(42)
np.random.seed(42)
torch.manual_seed(42)

print("Imports ready.")

In [ ]:
# ============================================
# CONFIGURATION
# ============================================

# Source dataset
ZIP_LOCAL = Path("/content/wlasl30.zip")
DATASET_ROOT = Path("/content/wlasl30")

# Output augmented dataset
AUGMENTED_ROOT = Path("/content/wlasl30_augmented")

# Number of augmented versions to create per original video
NUM_AUGMENTED_VERSIONS = 3

# Unzip original if needed
if not DATASET_ROOT.exists():
    if ZIP_LOCAL.exists():
        print(f"📦 Extracting {ZIP_LOCAL}...")
        !unzip -q {ZIP_LOCAL} -d /content/
        print("✅ Done.")
    else:
        raise FileNotFoundError(f"Upload {ZIP_LOCAL} first!")
else:
    print("✅ Source dataset ready.")

print(f"\n📊 Configuration:")
print(f"   Source: {DATASET_ROOT}")
print(f"   Output: {AUGMENTED_ROOT}")
print(f"   Augmented versions per video: {NUM_AUGMENTED_VERSIONS}")

## 1. Define Safe Augmentation Transforms

### Key Principle: Consistency Across Frames

For spatial augmentations (crop, rotation), we must apply the **same** transform to all frames in a video. Otherwise, the hand would "jump around" between frames.

For color augmentations, slight variation per frame is acceptable (simulates natural lighting changes).

In [ ]:
class VideoAugmentor:
    """
    Safe augmentation pipeline for ASL videos.
    
    Key features:
    - Spatial transforms are consistent across all frames
    - Color transforms can vary slightly per frame
    - No dangerous transforms (flip, strong rotation, time reversal)
    """
    
    def __init__(
        self,
        # Spatial augmentation params
        crop_scale: Tuple[float, float] = (0.85, 1.0),  # Random zoom 85-100%
        crop_ratio: Tuple[float, float] = (0.95, 1.05),  # Slight aspect ratio change
        rotation_degrees: float = 5.0,  # Max rotation ±5°
        
        # Color augmentation params
        brightness: float = 0.3,  # ±30% brightness
        contrast: float = 0.3,    # ±30% contrast
        saturation: float = 0.3,  # ±30% saturation
        hue: float = 0.05,        # ±5% hue (small to avoid unnatural colors)
        
        # Temporal augmentation params
        frame_drop_prob: float = 0.1,  # Probability of dropping each frame
        min_frames: int = 8,  # Minimum frames to keep
        
        # Output size
        output_size: int = 224
    ):
        self.crop_scale = crop_scale
        self.crop_ratio = crop_ratio
        self.rotation_degrees = rotation_degrees
        
        self.brightness = brightness
        self.contrast = contrast
        self.saturation = saturation
        self.hue = hue
        
        self.frame_drop_prob = frame_drop_prob
        self.min_frames = min_frames
        self.output_size = output_size
    
    def _get_spatial_params(self, img_size: Tuple[int, int]) -> dict:
        """
        Generate random spatial transform parameters.
        These will be applied consistently to ALL frames.
        """
        width, height = img_size
        
        # Random crop parameters
        scale = random.uniform(*self.crop_scale)
        ratio = random.uniform(*self.crop_ratio)
        
        # Calculate crop size
        crop_height = int(height * scale)
        crop_width = int(crop_height * ratio)
        crop_width = min(crop_width, width)
        crop_height = min(crop_height, height)
        
        # Random crop position
        top = random.randint(0, height - crop_height)
        left = random.randint(0, width - crop_width)
        
        # Random rotation
        angle = random.uniform(-self.rotation_degrees, self.rotation_degrees)
        
        return {
            'crop': (top, left, crop_height, crop_width),
            'angle': angle
        }
    
    def _get_color_params(self) -> dict:
        """
        Generate random color transform parameters.
        Called once per video (consistent) or per frame (variable).
        """
        return {
            'brightness': 1.0 + random.uniform(-self.brightness, self.brightness),
            'contrast': 1.0 + random.uniform(-self.contrast, self.contrast),
            'saturation': 1.0 + random.uniform(-self.saturation, self.saturation),
            'hue': random.uniform(-self.hue, self.hue)
        }
    
    def _apply_spatial_transform(self, img: Image.Image, params: dict) -> Image.Image:
        """Apply spatial transforms to a single frame."""
        top, left, crop_height, crop_width = params['crop']
        
        # Crop
        img = TF.crop(img, top, left, crop_height, crop_width)
        
        # Rotate (with white fill to avoid black edges)
        if abs(params['angle']) > 0.1:
            img = TF.rotate(img, params['angle'], fill=(128, 128, 128))
        
        # Resize to output size
        img = TF.resize(img, (self.output_size, self.output_size))
        
        return img
    
    def _apply_color_transform(self, img: Image.Image, params: dict) -> Image.Image:
        """Apply color transforms to a single frame."""
        img = TF.adjust_brightness(img, params['brightness'])
        img = TF.adjust_contrast(img, params['contrast'])
        img = TF.adjust_saturation(img, params['saturation'])
        img = TF.adjust_hue(img, params['hue'])
        
        return img
    
    def _apply_frame_dropping(self, frames: List[Image.Image]) -> List[Image.Image]:
        """
        Randomly drop frames to simulate variable video speed.
        Always keeps at least min_frames.
        """
        if len(frames) <= self.min_frames:
            return frames
        
        # Decide which frames to keep
        keep_mask = [random.random() > self.frame_drop_prob for _ in frames]
        
        # Always keep first and last frame
        keep_mask[0] = True
        keep_mask[-1] = True
        
        # Ensure minimum frames
        kept_frames = [f for f, keep in zip(frames, keep_mask) if keep]
        
        if len(kept_frames) < self.min_frames:
            # If too few frames, keep more
            return frames
        
        return kept_frames
    
    def augment_video(self, frames: List[Image.Image], 
                      apply_spatial: bool = True,
                      apply_color: bool = True,
                      apply_temporal: bool = True) -> List[Image.Image]:
        """
        Apply augmentation to a video (list of frames).
        
        Returns list of augmented PIL Images.
        """
        if len(frames) == 0:
            return frames
        
        # Get image size from first frame
        img_size = frames[0].size  # (width, height)
        
        # Generate consistent spatial params for entire video
        spatial_params = self._get_spatial_params(img_size) if apply_spatial else None
        
        # Generate color params (consistent across video for more natural look)
        color_params = self._get_color_params() if apply_color else None
        
        # Apply frame dropping first (before other transforms)
        if apply_temporal:
            frames = self._apply_frame_dropping(frames)
        
        # Apply transforms to each frame
        augmented_frames = []
        for frame in frames:
            aug_frame = frame.copy()
            
            if spatial_params:
                aug_frame = self._apply_spatial_transform(aug_frame, spatial_params)
            
            if color_params:
                aug_frame = self._apply_color_transform(aug_frame, color_params)
            
            augmented_frames.append(aug_frame)
        
        return augmented_frames


# Create augmentor instance
augmentor = VideoAugmentor(
    crop_scale=(0.80, 1.0),      # 80-100% zoom
    crop_ratio=(0.95, 1.05),     # Slight aspect ratio variation
    rotation_degrees=5.0,         # Max ±5° rotation
    brightness=0.3,               # ±30% brightness
    contrast=0.3,                 # ±30% contrast  
    saturation=0.3,               # ±30% saturation
    hue=0.05,                     # ±5% hue
    frame_drop_prob=0.1,          # 10% chance to drop each frame
    min_frames=8,                 # Keep at least 8 frames
    output_size=224               # Output 224x224
)

print("✅ VideoAugmentor created")

## 2. Visualize Augmentation Examples

In [ ]:
def load_video_frames(video_dir: Path) -> List[Image.Image]:
    """Load all frames from a video directory."""
    frame_files = sorted(video_dir.glob("*.jpg"))
    frames = [Image.open(f).convert("RGB") for f in frame_files]
    return frames


def visualize_augmentation(video_dir: Path, augmentor: VideoAugmentor, 
                           num_versions: int = 3, frames_to_show: int = 4):
    """
    Visualize original vs augmented versions of a video.
    """
    frames = load_video_frames(video_dir)
    
    if len(frames) == 0:
        print(f"No frames in {video_dir}")
        return
    
    # Sample frames to display
    indices = np.linspace(0, len(frames)-1, frames_to_show, dtype=int)
    sample_frames = [frames[i] for i in indices]
    
    # Create figure
    fig, axes = plt.subplots(num_versions + 1, frames_to_show, 
                             figsize=(3*frames_to_show, 3*(num_versions+1)))
    
    # Show original
    for j, frame in enumerate(sample_frames):
        axes[0, j].imshow(frame)
        axes[0, j].axis('off')
        if j == 0:
            axes[0, j].set_title('Original', fontsize=10)
    
    # Show augmented versions
    for i in range(num_versions):
        aug_frames = augmentor.augment_video(frames)
        aug_indices = np.linspace(0, len(aug_frames)-1, frames_to_show, dtype=int)
        aug_sample = [aug_frames[k] for k in aug_indices]
        
        for j, frame in enumerate(aug_sample):
            axes[i+1, j].imshow(frame)
            axes[i+1, j].axis('off')
            if j == 0:
                axes[i+1, j].set_title(f'Aug {i+1}', fontsize=10)
    
    plt.suptitle(f"Video: {video_dir.parent.name}/{video_dir.name}", fontsize=12)
    plt.tight_layout()
    plt.show()

In [ ]:
# Load a sample video and visualize augmentations
sample_videos = list((DATASET_ROOT / "train").glob("*/*/"))[:3]

print("📊 Augmentation Visualization")
print("="*60)

for video_dir in sample_videos:
    visualize_augmentation(video_dir, augmentor, num_versions=3, frames_to_show=5)

## 3. Generate Augmented Dataset

In [ ]:
def save_frames(frames: List[Image.Image], output_dir: Path):
    """Save frames to a directory."""
    output_dir.mkdir(parents=True, exist_ok=True)
    
    for i, frame in enumerate(frames):
        frame_path = output_dir / f"frame_{i:04d}.jpg"
        frame.save(frame_path, quality=95)


def generate_augmented_dataset(
    source_root: Path,
    output_root: Path,
    augmentor: VideoAugmentor,
    num_versions: int = 3,
    splits: List[str] = ["train"]  # Only augment training set
):
    """
    Generate augmented dataset.
    
    For each video in the source dataset:
    1. Copy the original video
    2. Create N augmented versions
    
    Structure:
    output_root/
    ├── train/
    │   ├── word1/
    │   │   ├── video1/         (original)
    │   │   ├── video1_aug1/    (augmented)
    │   │   ├── video1_aug2/    (augmented)
    │   │   └── ...
    ├── val/   (copied without augmentation)
    └── test/  (copied without augmentation)
    """
    
    # Load metadata
    with open(source_root / "metadata.json", "r") as f:
        metadata = json.load(f)
    
    # Create output directory
    if output_root.exists():
        print(f"⚠️ Removing existing {output_root}")
        shutil.rmtree(output_root)
    output_root.mkdir(parents=True)
    
    # Copy metadata
    with open(output_root / "metadata.json", "w") as f:
        json.dump(metadata, f, indent=2)
    
    stats = {
        'original_videos': 0,
        'augmented_videos': 0,
        'total_videos': 0
    }
    
    # Process each split
    for split in ["train", "val", "test"]:
        split_dir = source_root / split
        if not split_dir.exists():
            continue
        
        output_split_dir = output_root / split
        output_split_dir.mkdir(parents=True, exist_ok=True)
        
        should_augment = split in splits
        
        print(f"\n📁 Processing {split} set {'(with augmentation)' if should_augment else '(copy only)'}...")
        
        # Get all videos
        videos = []
        for gloss_dir in sorted(split_dir.iterdir()):
            if not gloss_dir.is_dir():
                continue
            for video_dir in sorted(gloss_dir.iterdir()):
                if video_dir.is_dir():
                    videos.append((gloss_dir.name, video_dir))
        
        for gloss, video_dir in tqdm(videos, desc=f"  {split}"):
            # Load original frames
            frames = load_video_frames(video_dir)
            
            if len(frames) == 0:
                continue
            
            # Create output gloss directory
            output_gloss_dir = output_split_dir / gloss
            output_gloss_dir.mkdir(parents=True, exist_ok=True)
            
            # 1. Save original video (resized to output_size for consistency)
            original_output_dir = output_gloss_dir / video_dir.name
            resized_frames = [TF.resize(f, (augmentor.output_size, augmentor.output_size)) for f in frames]
            save_frames(resized_frames, original_output_dir)
            stats['original_videos'] += 1
            stats['total_videos'] += 1
            
            # 2. Generate augmented versions (only for training)
            if should_augment:
                for aug_idx in range(num_versions):
                    aug_frames = augmentor.augment_video(frames)
                    aug_output_dir = output_gloss_dir / f"{video_dir.name}_aug{aug_idx+1}"
                    save_frames(aug_frames, aug_output_dir)
                    stats['augmented_videos'] += 1
                    stats['total_videos'] += 1
    
    return stats

In [ ]:
# Generate augmented dataset
print("="*60)
print("GENERATING AUGMENTED DATASET")
print("="*60)
print(f"\nSource: {DATASET_ROOT}")
print(f"Output: {AUGMENTED_ROOT}")
print(f"Augmented versions per video: {NUM_AUGMENTED_VERSIONS}")
print(f"Expected total training videos: original × {1 + NUM_AUGMENTED_VERSIONS}")

stats = generate_augmented_dataset(
    source_root=DATASET_ROOT,
    output_root=AUGMENTED_ROOT,
    augmentor=augmentor,
    num_versions=NUM_AUGMENTED_VERSIONS,
    splits=["train"]  # Only augment training set
)

print("\n" + "="*60)
print("GENERATION COMPLETE")
print("="*60)
print(f"\n📊 Statistics:")
print(f"   Original videos: {stats['original_videos']}")
print(f"   Augmented videos: {stats['augmented_videos']}")
print(f"   Total videos: {stats['total_videos']}")
print(f"   Expansion factor: {stats['total_videos'] / stats['original_videos']:.1f}x")

## 4. Verify Augmented Dataset

In [ ]:
# Count videos in augmented dataset
def count_dataset(root: Path) -> dict:
    """Count videos per split."""
    counts = {}
    for split in ["train", "val", "test"]:
        split_dir = root / split
        if not split_dir.exists():
            counts[split] = 0
            continue
        
        count = 0
        for gloss_dir in split_dir.iterdir():
            if gloss_dir.is_dir():
                count += len([d for d in gloss_dir.iterdir() if d.is_dir()])
        counts[split] = count
    
    return counts

# Compare original vs augmented
original_counts = count_dataset(DATASET_ROOT)
augmented_counts = count_dataset(AUGMENTED_ROOT)

print("\n📊 Dataset Comparison:")
print("="*50)
print(f"{'Split':<10} {'Original':<12} {'Augmented':<12} {'Factor':<10}")
print("-"*50)
for split in ["train", "val", "test"]:
    orig = original_counts[split]
    aug = augmented_counts[split]
    factor = aug / orig if orig > 0 else 0
    print(f"{split:<10} {orig:<12} {aug:<12} {factor:.1f}x")
print("="*50)
print(f"{'Total':<10} {sum(original_counts.values()):<12} {sum(augmented_counts.values()):<12}")

## 5. Create ZIP for Colab Transfer

In [ ]:
# Create ZIP file
ZIP_OUTPUT = Path("/content/wlasl30_augmented.zip")

print("📦 Creating ZIP file...")
!cd /content && zip -r -q wlasl30_augmented.zip wlasl30_augmented

# Get file size
zip_size = ZIP_OUTPUT.stat().st_size / (1024 * 1024)  # MB
print(f"✅ ZIP created: {ZIP_OUTPUT}")
print(f"   Size: {zip_size:.1f} MB")

In [ ]:
# Copy to Google Drive for persistence
DRIVE_OUTPUT = Path("/content/drive/MyDrive/tfg/datasets/wlasl30_augmented.zip")
DRIVE_OUTPUT.parent.mkdir(parents=True, exist_ok=True)

print(f"💾 Copying to Google Drive...")
shutil.copy(ZIP_OUTPUT, DRIVE_OUTPUT)
print(f"✅ Saved to: {DRIVE_OUTPUT}")

## 6. Summary

### What We Created

An augmented dataset with safe transformations that:
- **Preserve sign meaning** (no flip, strong rotation, or time reversal)
- **Increase diversity** (crop, color, frame dropping)
- **Are consistent across frames** (spatial transforms)

### Dataset Expansion

- Original training videos: ~245
- After augmentation: ~980 (4x expansion with 3 augmented versions)

### Next Steps

1. **Re-run variability analysis** (notebook 05) on augmented dataset
2. **Train model** (notebook 04) on augmented dataset
3. **Compare results** to baseline

In [ ]:
# Final summary
print("\n" + "="*60)
print("AUGMENTATION COMPLETE")
print("="*60)

print("\n🎯 Safe Augmentations Applied:")
print("   ✅ Random Crop/Zoom (80-100%)")
print("   ✅ Small Rotation (±5°)")
print("   ✅ Color Jitter (brightness, contrast, saturation)")
print("   ✅ Frame Dropping (10% probability)")

print("\n🚫 Dangerous Augmentations Avoided:")
print("   ❌ Horizontal Flip")
print("   ❌ Strong Rotation (>15°)")
print("   ❌ Time Reversal")

print("\n📁 Output Files:")
print(f"   Local: {AUGMENTED_ROOT}")
print(f"   ZIP: {ZIP_OUTPUT}")
print(f"   Drive: {DRIVE_OUTPUT}")

print("\n📊 To measure new variability, run notebook 05 with:")
print(f"   DATASET_ROOT = Path('{AUGMENTED_ROOT}')")